# Day 2 (teammate B) - ISL words (needs the Day-1 output dataset)
Inputs: `slr-code` and the Day-1 output (a dataset or a notebook output that contains `include50/`). Internet ON. CPU is fine (5-15 minutes); GPU is optional.

In [ ]:
# Finds the uploaded "slr-code" dataset wherever Kaggle mounts it (the path differs between Kaggle versions).
import glob, os, shutil, subprocess, sys, zipfile
ON_KAGGLE = os.path.exists("/kaggle/input")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
CODE = f"{WORK}/code"
hits = [h for d in ("*", "*/*", "*/*/*") for h in glob.glob(f"/kaggle/input/{d}/slr/__init__.py")] + glob.glob("/content/slr_code/**/slr/__init__.py", recursive=True)  # shallow search: ASL Alphabet has ~87k files
if hits:
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE, dirs_exist_ok=True)
else:
    zips = [h for d in ("*", "*/*", "*/*/*") for h in glob.glob(f"/kaggle/input/{d}/slr_code.zip")] + glob.glob("/content/**/slr_code.zip", recursive=True)
    if zips:
        zipfile.ZipFile(zips[0]).extractall(CODE)
    else:
        print("slr-code NOT FOUND. Add it: right sidebar -> + Add Input -> Your Datasets -> slr-code. Currently attached:")
        for p in sorted(glob.glob("/kaggle/input/*") + glob.glob("/kaggle/input/*/*")):
            print("  ", p)
        raise SystemExit("Attach the slr-code dataset, then run this cell again.")
os.chdir(CODE)
print("code ready in", CODE, os.listdir(CODE))
# Install each package on its own so one failure cannot skip the others. Kaggle now runs Python 3.13, so we use the
# current mediapipe (Tasks API) - not the old pinned version.
for pkg in ("mediapipe", "remotezip", "opencv-python-headless"):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], capture_output=True, text=True)
    print(pkg, "OK" if r.returncode == 0 else "FAILED\n" + r.stderr[-600:])
import mediapipe; print("mediapipe", mediapipe.__version__)

In [ ]:
import glob, os
def find_dir(marker, root="/kaggle/input", depth=5):
    """Folder that directly contains `marker` (file or folder), searched shallowly - Kaggle mount paths differ between versions."""
    for d in range(depth + 1):
        for p in glob.glob(root + "/*" * d + "/" + marker):
            return os.path.dirname(p)
    return None

## 1. Find the Day-1 keypoints

In [ ]:
INCLUDE_KP = find_dir("include50/train")
assert INCLUDE_KP, "Day-1 output not attached: + Add Input -> your Day-1 dataset / notebook output (must contain include50/)"
INCLUDE_KP = INCLUDE_KP + "/include50" if not INCLUDE_KP.endswith("include50") else INCLUDE_KP
print("keypoints:", INCLUDE_KP)
for s in ("train", "val", "test"):
    print(s, len(glob.glob(f"{INCLUDE_KP}/{s}/*.npz")), "files")

## 2. Train (about 5-15 minutes)

In [ ]:
!python scripts/train_words.py --data $INCLUDE_KP --out $WORK/models/isl_words.pt --epochs 150 --seeds 3

## 3. Package for download (a few MB)

In [ ]:
!cd $WORK && zip -r isl_models.zip models && ls -lh isl_models.zip

Then **Save Version -> Save & Run All**, open the Output tab and download `isl_models.zip`.